# Smarter Subrogation: Model Development Walkthrough

This notebook walks through the modeling workflow using the reusable modules
in `src/`: data-quality checks, feature engineering, LightGBM + Optuna,
calibration, and 1-SE threshold selection.

Run `python scripts/run_eda.py`, `train.py`, and `predict.py` for the
production CLI versions of this same workflow.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from config import CONFIG, NUMERIC_FEATURES, CATEGORICAL_FEATURES, PATHS, PLOT_DIR

df = pd.read_csv(PATHS["train"])
df.head()

## 1. Data quality: the "time travel glitches"

In [ ]:
from src.data_quality import (
    liability_vs_subrogation_rate, vehicle_year_glitch,
    license_age_issue, driver_age_anomalies, day_of_week_mismatch,
)

liability_vs_subrogation_rate(df, target_col=CONFIG["target_col"])

In [ ]:
vehicle_year_glitch(df)
driver_age_anomalies(df)
license_age_issue(df, save_path=PLOT_DIR / "license_age_issue.png")
day_of_week_mismatch(df, save_path=PLOT_DIR / "dow_confusion_matrix.png")
None

## 2. Feature engineering + preprocessing

`FeatureEngineer` applies the fixes above: driver age capped to [15, 100],
`invalid_vehicle_year` / `invalid_license_age` flags, day-of-week recomputed
from `claim_date`, and the financial-risk-ratio / threshold features.

In [ ]:
from src.pipeline import MLPipeline

pipeline = MLPipeline(
    df,
    target_col=CONFIG["target_col"],
    id_col=CONFIG["id_col"],
    numeric_features=NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    random_state=CONFIG["random_state"],
)
X, y = pipeline.prepare_data()

## 3. Optuna hyperparameter search

Tunes LightGBM's tree structure/regularization, `scale_pos_weight` for the
class imbalance, and the decision threshold jointly, using 10-fold CV
F1 as the objective. This cell can take a while at 100 trials; reduce
`n_trials` for a quick smoke test.

In [ ]:
study = pipeline.optimize(n_trials=CONFIG["optuna_trials"], cv_folds=CONFIG["optuna_cv_folds"])

## 4. Train the final model and inspect feature importance

In [ ]:
best_params = study.best_params.copy()
_, X_pre = pipeline.train_final_model(best_params)
pipeline.plot_feature_importance(top_n=20, save_path=PATHS["feature_importance_plot"])

## 5. Calibrate probabilities

A raw LightGBM score is optimized for ranking, not for "percent chance of
recovery." Isotonic regression (fit out-of-fold via `CalibratedClassifierCV`)
turns it into a number a business stakeholder can trust.

In [ ]:
from src.calibration import ModelCalibrator

calibrator = ModelCalibrator(method=CONFIG["calibration_method"], cv=CONFIG["calibration_cv"],
                              random_state=CONFIG["random_state"])
calibrator.calibrate(pipeline.best_model, X_pre, pipeline.y)
calibration_results = calibrator.evaluate(X_pre, pipeline.y)
calibrator.plot_calibration_curve(X_pre, pipeline.y, save_path=PATHS["calibration_plot"])

## 6. Threshold selection: the 1-SE rule

Rather than the raw F1-maximizing threshold, repeated stratified CV
(5 splits x 5 repeats = 25 iterations) plus the 1-SE rule selects a more
conservative, stable threshold.

In [ ]:
from src.threshold import ThresholdOptimizer

optimizer = ThresholdOptimizer(
    n_splits=CONFIG["threshold_cv_splits"],
    n_repeats=CONFIG["threshold_cv_repeats"],
    random_state=CONFIG["random_state"],
    threshold_range=CONFIG["threshold_range"],
    threshold_step=CONFIG["threshold_step"],
)
optimizer.optimize(X_pre, pipeline.y, calibrator.calibrated_model, refit_per_fold=False)
optimizer.plot(save_path=PATHS["threshold_plot"])
final_threshold = optimizer.get_threshold(option=CONFIG["threshold_selection"])
final_threshold

## 7. Save the model package and score the test set

In [ ]:
from src.model_io import save_model_package, load_model_package, predict_with_package

save_model_package(
    pipeline=pipeline,
    calibrator=calibrator,
    threshold=final_threshold,
    best_params=best_params,
    cv_f1_score=study.best_value,
    filename=PATHS["model_package"],
)

In [ ]:
df_test = pd.read_csv(PATHS["test"])
claim_ids = df_test[CONFIG["id_col"]]
X_test = df_test.drop(columns=[CONFIG["id_col"]], errors="ignore")

package = load_model_package(PATHS["model_package"])
preds, probs = predict_with_package(package, X_test)

submission = pd.DataFrame({CONFIG["id_col"]: claim_ids, CONFIG["target_col"]: preds})
submission.to_csv(PATHS["submission"], index=False)
submission.head()